In [3]:
import os
import re
import pandas as pd
import numpy as np
import warnings
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore")


# DB CONFIG

DB_CONFIG = {
    "drivername": "postgresql+psycopg2",
    "username": "postgres",
    "password": "Volkswagengolf2025",
    "host": "localhost",
    "port": 5432,
    "database": "NML_db",
}

DB_URL = (
    f"{DB_CONFIG['drivername']}://{DB_CONFIG['username']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)


# BASIC HELPERS

def safe_numeric(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    s = str(value).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return None
    s = s.replace(",", "")
    try:
        return float(s)
    except Exception:
        return None


def safe_date_convert(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    try:
        return pd.to_datetime(value).date()
    except Exception:
        s = str(value).strip()
        if s in ("", "nan", "NaN", "None"):
            return None
        try:
            return pd.to_datetime(s).date()
        except Exception:
            return None


def clean_value(val):
    if val is None:
        return None
    s = str(val).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return None
    return s



# ADMIN SHEET LABEL HELPERS (works for your "Admin" layout)

def _cell(df, r, c):
    try:
        v = df.iat[r, c]
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return None
        s = str(v).strip()
        return None if s in ("", "nan", "NaN", "None") else s
    except Exception:
        return None


def _find_label_positions(df, label_regex):
    hits = []
    rx = re.compile(label_regex, re.IGNORECASE)
    for r in range(df.shape[0]):
        for c in range(df.shape[1]):
            v = df.iat[r, c]
            if isinstance(v, str) and rx.search(v.strip()):
                hits.append((r, c))
    return hits


def _value_right(df, r, c, max_look=6):
    for k in range(1, max_look + 1):
        val = _cell(df, r, c + k)
        if val:
            return val
    return None


def _value_below(df, r, c, max_look=6):
    for k in range(1, max_look + 1):
        val = _cell(df, r + k, c)
        if val:
            return val
    return None


def get_labeled_value(df, label_regex, right_look=6, below_look=6):
    # match label that occupies a cell by itself (your Admin does this)
    hits = _find_label_positions(df, rf"^{label_regex}\s*:?\s*$")
    for (r, c) in hits:
        val = _value_right(df, r, c, right_look)
        if not val:
            val = _value_below(df, r, c, below_look)
        if val:
            return val
    return None


def get_block_below(df, label_regex, below_rows=8):
    hits = _find_label_positions(df, rf"^{label_regex}\s*:?\s*$")
    for (r, c) in hits:
        parts = []
        for k in range(1, below_rows + 1):
            v = _cell(df, r + k, c)
            if v:
                parts.append(v)
        if parts:
            return " ".join(parts)
    return None


def extract_admin_df(file_path):
    xl = pd.ExcelFile(file_path)
    sheet_name = "Admin" if "Admin" in xl.sheet_names else xl.sheet_names[0]
    return pd.read_excel(file_path, sheet_name=sheet_name, header=None, dtype=object)



# METADATA 

def extract_metadata_pf(file_path):
    """
    Common metadata only:
      - Uses Start Date as calibration_date (PF templates often have Start/End Date)
      - Leaves mass-only fields to mass_metadata (not touched here)
    """
    df = extract_admin_df(file_path)
    md = {"file_name": os.path.basename(file_path)}

    md["certificate_number"] = clean_value(
        get_labeled_value(df, r"Certificate\s*Number")
        or get_labeled_value(df, r"Cert(\.|ificate)?\s*No")
    )

    # PF Admin commonly uses Start Date / End Date
    md["calibration_date"] = safe_date_convert(
        get_labeled_value(df, r"Calibration\s*Date")
        or get_labeled_value(df, r"Mean\s*Date")
        or get_labeled_value(df, r"Start\s*Date")
        or get_labeled_value(df, r"End\s*Date")
    )

    md["manufacturer"] = clean_value(get_labeled_value(df, r"Manufacturer"))
    md["instrument_type"] = clean_value(get_labeled_value(df, r"Instrument\s*Type"))
    md["serial_number"] = clean_value(get_labeled_value(df, r"Serial\s*Number"))
    md["previous_certificate_number"] = clean_value(get_labeled_value(df, r"Previous\s*Certificate\s*Number"))
    md["date_received"] = safe_date_convert(get_labeled_value(df, r"Date\s*Received"))

    md["calibration_procedure_number"] = clean_value(
        get_labeled_value(df, r"Calibration\s*Procedure\s*Numbers?")
        or get_labeled_value(df, r"(NML\s+)?Procedure\s*Number")
    )

    md["method_used"] = clean_value(
        get_labeled_value(df, r"Method")
        or get_labeled_value(df, r"Description of Method Used")
    )

    md["comments"] = clean_value(
        get_labeled_value(df, r"Comments") or get_block_below(df, r"Comments", below_rows=10)
    )

    md["lab_type"] = "Pressure & Force"
    return md


def extract_instrument_details_pf(file_path):
    """
    For PF, we store Instrument Details (often used like a gauge/instrument ID).
    """
    try:
        df = extract_admin_df(file_path)
        return clean_value(get_labeled_value(df, r"(Instrument\s*Details|Instrument\s*ID|Gauge\s*ID)"))
    except Exception:
        return None



# RESULTS PARSING (Results (As Found) / Results (Adjusted))

def _find_pf_header_row(df_raw):
    """
    Finds the row where the main results table header begins (contains Point + Reference + Instrument).
    """
    for r in range(min(250, df_raw.shape[0])):
        row = [str(v).lower() if isinstance(v, str) else "" for v in df_raw.iloc[r]]
        joined = " ".join(row)
        if ("point" in joined) and ("reference" in joined) and ("instrument" in joined):
            return r
    return None


def _unit_from_cell(cell):
    if not isinstance(cell, str):
        return None
    s = cell.strip()
    # e.g. "(bar)" -> "bar"
    m = re.search(r"\(([^)]+)\)", s)
    return m.group(1).strip() if m else None


def _to_bool(val):
    if isinstance(val, bool):
        return val
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return None
    s = str(val).strip().lower()
    if s in {"true", "t", "yes", "y", "pass"}:
        return True
    if s in {"false", "f", "no", "n", "fail"}:
        return False
    return None


def parse_pf_results_sheet(file_path, sheet_name, condition):
    """
    Parses a PF results sheet into records.

    condition: 'As Found' or 'Adjusted'

    Extracts:
      - point
      - reference_pressure
      - instrument_reading
      - deviation
      - standard (text)
      - generated_uncertainty (same unit as reference)
      - tolerance, decision_rule, range_check, uut_resolution
      - plus a few sheet-level fields: medium_used, gauge_position, temperature, range, resolution, units
    """
    try:
        df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None, dtype=object)
    except Exception as e:
        print(f"   ⚠️ Could not read sheet '{sheet_name}': {e}")
        return pd.DataFrame()

    # Pull a few sheet-level fields (top area)
    cert_no = None
    date_val = None
    medium_used = None
    gauge_position = None
    temperature = None
    range_box = None
    resolution_box = None
    units_box = None

    # scan top rows for key labels
    for r in range(min(30, df_raw.shape[0])):
        row = df_raw.iloc[r].tolist()
        for c, v in enumerate(row):
            if not isinstance(v, str):
                continue
            low = v.strip().lower()
            if low == "cert no.:":
                cert_no = df_raw.iat[r, c + 1] if c + 1 < df_raw.shape[1] else None
            elif low == "date:":
                date_val = df_raw.iat[r, c + 1] if c + 1 < df_raw.shape[1] else None
            elif low == "medium used:":
                medium_used = df_raw.iat[r, c + 1] if c + 1 < df_raw.shape[1] else None
            elif low == "gauge position:":
                gauge_position = df_raw.iat[r, c + 1] if c + 1 < df_raw.shape[1] else None
            elif low == "temperature:":
                temperature = df_raw.iat[r, c + 1] if c + 1 < df_raw.shape[1] else None
            elif low == "range:":
                range_box = df_raw.iat[r, c + 1] if c + 1 < df_raw.shape[1] else None
            elif low == "resolution:":
                resolution_box = df_raw.iat[r, c + 1] if c + 1 < df_raw.shape[1] else None
            elif low == "units:":
                units_box = df_raw.iat[r, c + 1] if c + 1 < df_raw.shape[1] else None

    header_row = _find_pf_header_row(df_raw)
    if header_row is None:
        print(f"   ⚠️ Could not find PF table header in '{sheet_name}'")
        return pd.DataFrame()

    header = [str(v).strip().lower() if isinstance(v, str) else "" for v in df_raw.iloc[header_row]]
    units_row = df_raw.iloc[header_row + 1] if header_row + 1 < df_raw.shape[0] else None

    def find_col(*keywords):
        for idx, col in enumerate(header):
            ok = True
            for kw in keywords:
                if kw not in col:
                    ok = False
                    break
            if ok:
                return idx
        return None

    col_point = find_col("point")
    col_ref = find_col("reference")
    col_inst = find_col("instrument")
    col_dev = find_col("deviation")
    col_std = find_col("standard")
    col_gen_unc = find_col("uncertainty", "generated")
    col_tol = find_col("tolerance")
    col_dec = find_col("decision")
    col_rangechk = find_col("range check")
    col_res = find_col("resolution")  # UUT resolution column
    col_meas_unc = find_col("unc.", "measured") or find_col("unc", "measured")

    if col_point is None or col_ref is None or col_inst is None:
        print(f"   ⚠️ Missing required PF columns in '{sheet_name}'")
        return pd.DataFrame()

    # Determine units from the units row (e.g. (bar), (mbar))
    ref_unit = _unit_from_cell(units_row.iloc[col_ref]) if units_row is not None else None
    inst_unit = _unit_from_cell(units_row.iloc[col_inst]) if units_row is not None else None
    dev_unit = _unit_from_cell(units_row.iloc[col_dev]) if (units_row is not None and col_dev is not None) else None
    unit = ref_unit or inst_unit or dev_unit or clean_value(units_box)

    records = []
    started = False

    for r in range(header_row + 2, df_raw.shape[0]):
        row = df_raw.iloc[r]

        point = safe_numeric(row.iloc[col_point]) if col_point is not None else None
        ref_val = safe_numeric(row.iloc[col_ref]) if col_ref is not None else None
        inst_val = safe_numeric(row.iloc[col_inst]) if col_inst is not None else None
        dev_val = safe_numeric(row.iloc[col_dev]) if col_dev is not None else None

        # stop when the main numeric columns go empty after we started
        if point is None and ref_val is None and inst_val is None and dev_val is None:
            if started:
                break
            continue
        started = True

        if dev_val is None and (ref_val is not None) and (inst_val is not None):
            dev_val = inst_val - ref_val

        std_text = clean_value(row.iloc[col_std]) if col_std is not None else None
        gen_unc = safe_numeric(row.iloc[col_gen_unc]) if col_gen_unc is not None else None
        tol = safe_numeric(row.iloc[col_tol]) if col_tol is not None else None
        decision = _to_bool(row.iloc[col_dec]) if col_dec is not None else None
        range_check = clean_value(row.iloc[col_rangechk]) if col_rangechk is not None else None
        uut_res = safe_numeric(row.iloc[col_res]) if col_res is not None else None
        meas_unc = safe_numeric(row.iloc[col_meas_unc]) if col_meas_unc is not None else None

        records.append({
            "certificate_number_in_sheet": clean_value(cert_no),
            "sheet_date": safe_date_convert(date_val),
            "condition": condition,
            "point": int(point) if point is not None and float(point).is_integer() else point,
            "reference_value": ref_val,
            "instrument_reading": inst_val,
            "deviation": dev_val,
            "unit": clean_value(unit),
            "standard": std_text,
            "generated_uncertainty": gen_unc,
            "tolerance": tol,
            "decision_rule": decision,
            "range_check": range_check,
            "uut_resolution": uut_res,
            "measured_uncertainty": meas_unc,
            "medium_used": clean_value(medium_used),
            "gauge_position": clean_value(gauge_position),
            "temperature_c": safe_numeric(temperature),
            "range_box": safe_numeric(range_box),
            "resolution_box": safe_numeric(resolution_box),
            "units_box": clean_value(units_box),
        })

    if not records:
        print(f"   ⚠️ No PF data rows found in '{sheet_name}'")
        return pd.DataFrame()

    print(f"   ✅ Extracted {len(records)} PF rows from '{sheet_name}' ({condition})")
    return pd.DataFrame(records)


def extract_pf_measurements(file_path):
    """
    Reads both:
      - Results (As Found)
      - Results (Adjusted)
    Returns a unified DataFrame.
    """
    try:
        xl = pd.ExcelFile(file_path)
    except Exception as e:
        print(f"   ❌ Cannot open workbook: {e}")
        return pd.DataFrame()

    dfs = []
    if "Results (As Found)" in xl.sheet_names:
        dfs.append(parse_pf_results_sheet(file_path, "Results (As Found)", condition="As Found"))
    if "Results (Adjusted)" in xl.sheet_names:
        dfs.append(parse_pf_results_sheet(file_path, "Results (Adjusted)", condition="Adjusted"))

    if not dfs:
        print("   ⚠️ No PF results sheets found")
        return pd.DataFrame()

    out = pd.concat([d for d in dfs if d is not None and not d.empty], ignore_index=True)
    return out if not out.empty else pd.DataFrame()



# SCHEMA: Ensure universal metadata + PF results table

def ensure_pf_schema(engine):
    """
    Ensures:
      - calibration_metadata has ONLY universal columns (NO mass-only columns)
      - pressure_force_results exists
      - uq_calibration_metadata_file_name exists
      - uq_pf_unique_point_per_condition exists
    """
    with engine.begin() as conn:
        # --- Universal metadata table ONLY (no mass fields here) ---
        conn.execute(text("""
        CREATE TABLE IF NOT EXISTS calibration_metadata (
            id SERIAL PRIMARY KEY,
            file_name TEXT,
            certificate_number TEXT,
            calibration_date DATE,

            instrument_type TEXT,
            manufacturer TEXT,
            serial_number TEXT,
            previous_certificate_number TEXT,
            date_received DATE,
            calibration_procedure_number TEXT,
            method_used TEXT,
            comments TEXT,

            lab_type TEXT
        );
        """))

        # Add missing universal columns safely
        for col, coltype in [
            ("file_name", "TEXT"),
            ("certificate_number", "TEXT"),
            ("calibration_date", "DATE"),
            ("instrument_type", "TEXT"),
            ("manufacturer", "TEXT"),
            ("serial_number", "TEXT"),
            ("previous_certificate_number", "TEXT"),
            ("date_received", "DATE"),
            ("calibration_procedure_number", "TEXT"),
            ("method_used", "TEXT"),
            ("comments", "TEXT"),
            ("lab_type", "TEXT"),
        ]:
            conn.execute(text(f"ALTER TABLE calibration_metadata ADD COLUMN IF NOT EXISTS {col} {coltype};"))

        # --- Dedupe constraint on metadata ---
        exists = conn.execute(text("""
            SELECT 1 FROM pg_constraint WHERE conname='uq_calibration_metadata_file_name'
        """)).scalar()
        if not exists:
            conn.execute(text("""
                ALTER TABLE calibration_metadata
                ADD CONSTRAINT uq_calibration_metadata_file_name UNIQUE (file_name);
            """))

        # --- PF results table ---
        conn.execute(text("""
        CREATE TABLE IF NOT EXISTS pressure_force_results (
            id SERIAL PRIMARY KEY,
            metadata_id INTEGER REFERENCES calibration_metadata(id) ON DELETE CASCADE,
            certificate_number TEXT,
            instrument_id TEXT,
            condition TEXT,
            point INTEGER,
            reference_value DOUBLE PRECISION,
            instrument_reading DOUBLE PRECISION,
            deviation DOUBLE PRECISION,
            unit TEXT,
            standard TEXT,
            generated_uncertainty DOUBLE PRECISION,
            tolerance DOUBLE PRECISION,
            decision_rule BOOLEAN,
            range_check TEXT,
            uut_resolution DOUBLE PRECISION,
            measured_uncertainty DOUBLE PRECISION,
            medium_used TEXT,
            gauge_position TEXT,
            temperature_c DOUBLE PRECISION,
            range_box DOUBLE PRECISION,
            resolution_box DOUBLE PRECISION,
            units_box TEXT
        );
        """))

        # Prevent duplicates per file/condition/point
        exists = conn.execute(text("""
            SELECT 1 FROM pg_constraint WHERE conname='uq_pf_unique_point_per_condition'
        """)).scalar()
        if not exists:
            conn.execute(text("""
                ALTER TABLE pressure_force_results
                ADD CONSTRAINT uq_pf_unique_point_per_condition
                UNIQUE (metadata_id, condition, point);
            """))


# DEDUPE / BACKFILL LOGIC
# - If metadata exists but PF results are missing -> backfill.
# - If results already exist -> skip.

def get_metadata_id(engine, file_name: str):
    with engine.connect() as conn:
        return conn.execute(
            text("SELECT id FROM calibration_metadata WHERE file_name=:fn LIMIT 1"),
            {"fn": file_name},
        ).scalar()


def pf_results_exist(engine, metadata_id: int) -> bool:
    with engine.connect() as conn:
        cnt = conn.execute(
            text("SELECT COUNT(*) FROM pressure_force_results WHERE metadata_id=:mid"),
            {"mid": metadata_id},
        ).scalar()
    return (cnt or 0) > 0


def insert_pf_file(engine, file_path):
    file_name = os.path.basename(file_path)

    # If metadata exists AND results exist -> skip
    mid = get_metadata_id(engine, file_name)
    if mid and pf_results_exist(engine, mid):
        print(f"⏭️  Skipping (already imported with PF results): {file_name}")
        return True

    print(f"\n{'='*60}")
    print(f"📄 {file_name}")
    print("=" * 60)

    # Extract metadata
    md = extract_metadata_pf(file_path)
    instrument_id = extract_instrument_details_pf(file_path)

    # Insert metadata if not present
    if not mid:
        try:
            pd.DataFrame([md]).to_sql("calibration_metadata", engine, if_exists="append", index=False)
            mid = get_metadata_id(engine, file_name)
            print(f"✅ Metadata inserted (ID: {mid}, lab_type=Pressure & Force)")
        except Exception as e:
            msg = str(e).lower()
            # if file_name unique constraint exists (from your earlier work), handle duplicates gracefully
            if "duplicate key value" in msg:
                mid = get_metadata_id(engine, file_name)
                print(f"🔁 Metadata already exists (ID: {mid}), continuing.")
            else:
                print(f"❌ Metadata insert error: {e}")
                return False
    else:
        print(f"🔁 Backfill mode: metadata already exists (ID: {mid}), inserting PF results.")

    cert_no = md.get("certificate_number")

    # Extract results
    df = extract_pf_measurements(file_path)
    if df.empty:
        print("⚠️ No PF results extracted (metadata stored).")
        return True

    # Attach FK + common fields
    df["metadata_id"] = mid
    df["certificate_number"] = cert_no
    df["instrument_id"] = instrument_id

    # Keep only columns that match the table
    keep = [
        "metadata_id", "certificate_number", "instrument_id",
        "condition", "point",
        "reference_value", "instrument_reading", "deviation", "unit",
        "standard", "generated_uncertainty", "tolerance", "decision_rule",
        "range_check", "uut_resolution", "measured_uncertainty",
        "medium_used", "gauge_position", "temperature_c",
        "range_box", "resolution_box", "units_box"
    ]
    df = df[keep]

    # Insert: because we have a UNIQUE(metadata_id, condition, point),
    # we insert row-by-row with ON CONFLICT DO NOTHING (pandas to_sql can't do that).
    insert_sql = text("""
        INSERT INTO pressure_force_results (
            metadata_id, certificate_number, instrument_id,
            condition, point,
            reference_value, instrument_reading, deviation, unit,
            standard, generated_uncertainty, tolerance, decision_rule,
            range_check, uut_resolution, measured_uncertainty,
            medium_used, gauge_position, temperature_c,
            range_box, resolution_box, units_box
        ) VALUES (
            :metadata_id, :certificate_number, :instrument_id,
            :condition, :point,
            :reference_value, :instrument_reading, :deviation, :unit,
            :standard, :generated_uncertainty, :tolerance, :decision_rule,
            :range_check, :uut_resolution, :measured_uncertainty,
            :medium_used, :gauge_position, :temperature_c,
            :range_box, :resolution_box, :units_box
        )
        ON CONFLICT (metadata_id, condition, point) DO NOTHING;
    """)

    try:
        with engine.begin() as conn:
            for _, row in df.iterrows():
                conn.execute(insert_sql, {k: (None if (isinstance(row[k], float) and np.isnan(row[k])) else row[k]) for k in df.columns})
        print(f"✅ Inserted {len(df)} PF rows into 'pressure_force_results'")
        return True
    except Exception as e:
        print(f"❌ PF insert error: {e}")
        return False



# MAIN

def run_pressure_force_import(folder_path=".", ensure_schema=True):
    """
    Put the PF .xlsm certificates in the SAME folder as this script,
    then run it. It will:
      - insert metadata into calibration_metadata
      - insert results into pressure_force_results
      - skip already-imported files (unless they have metadata but missing results -> backfill)
    """
    print("🚀 NML Pressure & Force Import Tool")
    print("=" * 70)

    try:
        engine = create_engine(DB_URL)
        with engine.connect() as conn:
            conn.execute(text("SELECT 1"))
        print("✅ Database connected\n")
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        return

    if ensure_schema:
        ensure_pf_schema(engine)

    files = [f for f in os.listdir(folder_path) if f.lower().endswith(".xlsm") and not f.startswith("~$")]
    if not files:
        print(f"⚠️ No .xlsm files found in '{folder_path}'")
        return

    print(f"Found {len(files)} file(s)\n")

    ok = 0
    for f in sorted(files):
        if insert_pf_file(engine, os.path.join(folder_path, f)):
            ok += 1

    print(f"\n{'='*70}")
    print("📊 SUMMARY")
    print("=" * 70)
    print(f"Seen in folder: {len(files)}")
    print(f"Handled successfully: {ok}")
    print(f"Errors: {len(files) - ok}")

    try:
        with engine.connect() as conn:
            meta_cnt = conn.execute(text("SELECT COUNT(*) FROM calibration_metadata")).scalar()
            pf_cnt = conn.execute(text("SELECT COUNT(*) FROM pressure_force_results")).scalar()
            print(f"   public.calibration_metadata: {meta_cnt} records")
            print(f"   public.pressure_force_results: {pf_cnt} records")
    except Exception as e:
        print(f"⚠️ Count error: {e}")

    print("\n✅ Pressure & Force import complete!")


if __name__ == "__main__":
    run_pressure_force_import(folder_path=".", ensure_schema=True)


🚀 NML Pressure & Force Import Tool
✅ Database connected

Found 5 file(s)


📄 24-7416.xlsm
✅ Metadata inserted (ID: 69, lab_type=Pressure & Force)
   ✅ Extracted 31 PF rows from 'Results (As Found)' (As Found)
   ✅ Extracted 31 PF rows from 'Results (Adjusted)' (Adjusted)
✅ Inserted 62 PF rows into 'pressure_force_results'

📄 24-7446.xlsm
✅ Metadata inserted (ID: 70, lab_type=Pressure & Force)
   ✅ Extracted 31 PF rows from 'Results (As Found)' (As Found)
   ✅ Extracted 31 PF rows from 'Results (Adjusted)' (Adjusted)
✅ Inserted 62 PF rows into 'pressure_force_results'

📄 24-7487.xlsm
✅ Metadata inserted (ID: 71, lab_type=Pressure & Force)
   ✅ Extracted 31 PF rows from 'Results (As Found)' (As Found)
   ✅ Extracted 31 PF rows from 'Results (Adjusted)' (Adjusted)
✅ Inserted 62 PF rows into 'pressure_force_results'

📄 24-7538.xlsm
✅ Metadata inserted (ID: 72, lab_type=Pressure & Force)
   ✅ Extracted 31 PF rows from 'Results (As Found)' (As Found)
   ✅ Extracted 31 PF rows from 'Results (